# Payer 360

In [0]:
%sql
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
%sql
SELECT COUNT(DISTINCT PATIENT_ID) AS PatientCount
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master;

-- SELECT *
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master;

In [0]:
%sql
-- ============================================================
-- 1. Base Payer Table
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_base AS
SELECT DISTINCT
    PAYER_ID,
    PAYER_NAME,
    PARENT_ID,
    PARENT_NAME,
    KH_PLAN_ID      AS PLAN_ID,
    INSURANCE_GROUP
FROM com_edp_prd.com_raw.kom_plans
WHERE PAYER_ID IS NOT NULL; 

SELECT * FROM payer_base LIMIT 10;

In [0]:
%sql
-- ============================================================
-- TOTAL LIVES BASE (1 row per patient)
-- Most-recent event drives KH_PLAN + HCP, then territory
-- Adds payer (insurance_group) from kom_plans (deduped)
-- ============================================================

CREATE OR REPLACE TEMP VIEW total_lives AS

WITH medical AS (
  SELECT
      PATIENT_ID,
      KH_PLAN_ID AS KH_PLAN,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
      SERVICE_DATE AS event_date
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

pharmacy AS (
  SELECT
      PATIENT_ID,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      PRESCRIBER_NPI AS HCP_NPI,
      FILL_DATE AS event_date
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

base AS (
  SELECT * FROM medical
  UNION ALL
  SELECT * FROM pharmacy
),

-- 1 row per patient: most recent event wins
latest_event AS (
  SELECT
    PATIENT_ID,
    KH_PLAN,
    HCP_NPI,
    event_date
  FROM (
    SELECT
      b.*,
      ROW_NUMBER() OVER (
        PARTITION BY PATIENT_ID
        ORDER BY event_date DESC, KH_PLAN DESC, HCP_NPI DESC
      ) AS rn
    FROM base b
    WHERE b.KH_PLAN IS NOT NULL
  ) x
  WHERE rn = 1
),

prov AS (
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP,''),'[^0-9]',''),1,5)
        AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
),

-- dedupe kom_plans to 1 row per KH_PLAN_ID (only fields we need)
kom_plans_1row AS (
  SELECT KH_PLAN_ID, insurance_group
  FROM (
    SELECT
      KH_PLAN_ID,
      insurance_group,
      ROW_NUMBER() OVER (
        PARTITION BY KH_PLAN_ID
        ORDER BY
          CASE WHEN insurance_group IS NULL THEN 1 ELSE 0 END,
          insurance_group DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_plans
  ) p
  WHERE rn = 1
),

-- (optional but defensive) dedupe zip mapping to 1 row per zipcode
zip_map_1row AS (
  SELECT
    zipcode,
    MAX(TRY_CAST(territory_id AS BIGINT)) AS territory_id,
    MAX(territory_name) AS territory_name,
    MAX(TRY_CAST(region_id AS BIGINT)) AS region_id,
    MAX(region_name) AS region_name
  FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
  GROUP BY zipcode
)

SELECT
  le.PATIENT_ID,
  le.KH_PLAN AS kh_plan_id,                    -- plan
  pl.insurance_group AS payer_insurance_group, -- payer
  COALESCE(z.territory_id, -2) AS territory_id,
  COALESCE(z.territory_name, 'UNKNOWN') AS territory_name,
  COALESCE(z.region_id, -2) AS region_id,
  COALESCE(z.region_name, 'UNKNOWN') AS region_name,
  le.event_date AS most_recent_event_date
FROM latest_event le
LEFT JOIN prov p
  ON TRIM(CAST(le.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN zip_map_1row z
  ON p.zip5_int = z.zipcode
LEFT JOIN kom_plans_1row pl
  ON le.KH_PLAN = pl.KH_PLAN_ID;

-- sanity: should match


In [0]:
select payer_insurance_group, kh_plan_id, count(*) from total_lives
group by payer_insurance_group, kh_plan_id
order by 2;

In [0]:
%skip
%sql
-- ============================================================
-- TOTAL LIVES BASE (ALL PATIENTS - NOT patient360 restricted)
-- Medical + PAID Pharmacy
-- ============================================================

CREATE OR REPLACE TEMP VIEW total_lives AS

WITH medical AS (
  SELECT DISTINCT
      PATIENT_ID,
      KH_PLAN_ID AS KH_PLAN,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

pharmacy AS (
  SELECT DISTINCT
      PATIENT_ID,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      PRESCRIBER_NPI AS HCP_NPI
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

base AS (
  SELECT * FROM medical
  UNION
  SELECT * FROM pharmacy
),

prov AS (
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP,''),'[^0-9]',''),1,5)
        AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
)

SELECT DISTINCT
  b.PATIENT_ID AS PATIENT_ID,
  b.KH_PLAN,
  COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
  COALESCE(z.territory_name, 'UNKNOWN') AS territory_name,
  COALESCE(TRY_CAST(z.region_id AS BIGINT), -2) AS region_id,
  COALESCE(z.region_name, 'UNKNOWN') AS region_name
FROM base b
LEFT JOIN prov p
  ON TRIM(CAST(b.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5_int = z.zipcode
WHERE b.KH_PLAN IS NOT NULL;


----
SELECT COUNT(*) AS n, COUNT(DISTINCT PATIENT_ID) AS n_distinct FROM total_lives;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS

WITH base AS (

    -- ============================================================
    -- 1️⃣ Medical (Elaprase NDC)
    -- ============================================================
    SELECT DISTINCT
        m.PATIENT_ID                              AS PATIENT_ID,
        m.RENDERING_NPI                           AS HCP_NPI,
        m.BILLING_NPI                             AS HCO_NPI,
        m.NDC11                                   AS CODE,
        m.MEDICAL_EVENT_ID                        AS EVENT_ID,
        m.SERVICE_DATE                            AS FILL_DATE,
        m.PLACE_OF_SERVICE                        AS PLACE_OF_SERVICE,
        m.KH_PLAN_ID                              AS KH_PLAN,
        NULL                                      AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                          AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON m.PATIENT_ID = pat.PATIENT_ID
    WHERE m.NDC11 IN ('54092070001','540920700')

    UNION

    -- ============================================================
    -- 2️⃣ Pharmacy (Elaprase NDC | PAID only)
    -- ============================================================
    SELECT DISTINCT
        ph.PATIENT_ID                              AS PATIENT_ID,
        ph.PRESCRIBER_NPI                          AS HCP_NPI,
        ph.PHARMACY_NPI                            AS HCO_NPI,
        ph.NDC11                                   AS CODE,
        ph.PHARMACY_EVENT_ID                       AS EVENT_ID,
        ph.FILL_DATE                               AS FILL_DATE,
        NULL                                       AS PLACE_OF_SERVICE,
        COALESCE(ph.PRIMARY_KH_PLAN_ID, ph.SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        ph.PHARMACY_CHANNEL                        AS PHARMACY_CHANNEL,
        'PHARMACY_EVENTS'                          AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events ph
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON ph.PATIENT_ID = pat.PATIENT_ID
    WHERE ph.NDC11 IN ('54092070001','540920700')
      AND ph.TRANSACTION_RESULT = 'PAID'

    UNION

    -- ============================================================
    -- 3️⃣ Medical (Elaprase Procedure Codes)
    -- ============================================================
    SELECT DISTINCT
        m.PATIENT_ID                              AS PATIENT_ID,
        m.RENDERING_NPI                           AS HCP_NPI,
        m.BILLING_NPI                             AS HCO_NPI,
        m.PROCEDURE_CODE                          AS CODE,
        m.MEDICAL_EVENT_ID                        AS EVENT_ID,
        m.SERVICE_DATE                            AS FILL_DATE,
        m.PLACE_OF_SERVICE                        AS PLACE_OF_SERVICE,
        m.KH_PLAN_ID                              AS KH_PLAN,
        NULL                                      AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                          AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON m.PATIENT_ID = pat.PATIENT_ID
    WHERE m.PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )

),

filtered AS (
    SELECT *
    FROM base
    WHERE FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

prov AS (
    SELECT
        TRIM(CAST(NPI AS STRING)) AS npi_str,
        MAX(
            TRY_CAST(
                SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP,''),'[^0-9]',''),1,5)
                AS BIGINT
            )
        ) AS zip5_int
    FROM com_edp_prd.com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
    GROUP BY TRIM(CAST(NPI AS STRING))
)

SELECT DISTINCT
    f.*,
    COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
    COALESCE(z.territory_name, 'UNKNOWN')            AS territory_name,
    COALESCE(TRY_CAST(z.region_id AS BIGINT), -2)   AS region_id,
    COALESCE(z.region_name, 'UNKNOWN')              AS region_name
FROM filtered f
LEFT JOIN prov p
    ON TRIM(CAST(f.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON p.zip5_int = z.zipcode;

-- Show table
SELECT * FROM MPSII_TREATMENT_TABLE LIMIT 100;


In [0]:
%skip
SELECT * FROM MPSII_TREATMENT_TABLE
WHERE EVENT_ID IN
(SELECT EVENT_ID FROM MPSII_TREATMENT_TABLE
GROUP BY EVENT_ID
HAVING COUNT(*) > 1)
ORDER BY EVENT_ID ASC;

In [0]:
SELECT
  PATIENT_ID,
  MAX(FILL_DATE) AS MAX_FILL_DATE,
  MAX_BY(KH_PLAN, FILL_DATE) AS KH_PLAN_AT_MAX_FILL_DATE
FROM MPSII_TREATMENT_TABLE
GROUP BY PATIENT_ID;

In [0]:
%skip

%sql
CREATE OR REPLACE TEMP VIEW elaprase_patient_payer AS
SELECT DISTINCT
  t.PATIENT_ID,
  p.PAYER_ID,
  t.territory_id,
  t.territory_name,
  t.region_id,
  t.region_name
FROM MPSII_TREATMENT_TABLE t
JOIN payer_base p
  ON t.KH_PLAN = p.PLAN_ID;

SELECT COUNT(PATIENT_ID), COUNT(DISTINCT PATIENT_ID) FROM elaprase_patient_payer;


In [0]:
%sql
-- ============================================================
-- Attribute each patient to ONE "latest KH_PLAN"
-- Meaning:
--   Pick the most recent event where we can identify KH_PLAN.
--   Prefer events whose KH_PLAN maps to payer_base (so payer is real).
-- Tie-breakers:
--   1) mappable plan > non-mappable (but non-null) plan
--   2) most recent FILL_DATE
--   3) Pharmacy over Medical on same day
--   4) deterministic tie-breaks
-- Output is 1 row per PATIENT_ID
-- ============================================================

CREATE OR REPLACE TEMP VIEW elaprase_patient_latest_khplan AS
WITH t AS (
  SELECT
    t.*,
    CASE WHEN pb.PLAN_ID IS NOT NULL THEN 1 ELSE 0 END AS is_mappable_plan
  FROM MPSII_TREATMENT_TABLE t
  LEFT JOIN payer_base pb
    ON t.KH_PLAN = pb.PLAN_ID
  WHERE t.FILL_DATE IS NOT NULL
    AND t.KH_PLAN IS NOT NULL
),
ranked AS (
  SELECT
    PATIENT_ID,
    KH_PLAN,
    FILL_DATE,
    TABLE_NAME,
    EVENT_ID,
    territory_id,
    territory_name,
    region_id,
    region_name,
    is_mappable_plan,
    ROW_NUMBER() OVER (
      PARTITION BY PATIENT_ID
      ORDER BY
        is_mappable_plan DESC,                              -- prefer mappable plan
        FILL_DATE DESC,                                     -- then most recent
        CASE WHEN TABLE_NAME = 'PHARMACY_EVENTS' THEN 1 ELSE 2 END,
        CAST(EVENT_ID AS STRING) DESC,
        CAST(KH_PLAN AS STRING) DESC
    ) AS rn
  FROM t
)
SELECT
  PATIENT_ID,
  KH_PLAN,
  FILL_DATE AS chosen_fill_date,
  TABLE_NAME AS chosen_source,
  EVENT_ID   AS chosen_event_id,
  territory_id,
  territory_name,
  region_id,
  region_name,
  is_mappable_plan
FROM ranked
WHERE rn = 1
;

-- sanity: must be 1:1
SELECT COUNT(*) AS n, COUNT(DISTINCT PATIENT_ID) AS n_distinct
FROM elaprase_patient_latest_khplan;

-- ============================================================
-- Build patient->payer using the chosen "latest KH_PLAN"
-- (Now you should be much closer to the 410 expectation.)
-- ============================================================

CREATE OR REPLACE TEMP VIEW elaprase_patient_payer AS
SELECT
  e.PATIENT_ID,
  p.PAYER_ID,
  e.territory_id,
  e.territory_name,
  e.region_id,
  e.region_name
FROM elaprase_patient_latest_khplan e
JOIN payer_base p
  ON e.KH_PLAN = p.PLAN_ID
;

-- Patient Counts
SELECT COUNT(*) AS NROWS, COUNT(DISTINCT PATIENT_ID) AS NPATS FROM elaprase_patient_payer;

In [0]:
%sql
-- payer_base duplicates can still explode patients
SELECT PLAN_ID, COUNT(*) n
FROM payer_base
GROUP BY PLAN_ID
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 50;

-- “latest KH_PLAN” vs “latest mappable KH_PLAN”
-- should be equal if 1 row per patient
SELECT
  (SELECT COUNT(*) FROM elaprase_patient_payer) AS rows,
  (SELECT COUNT(DISTINCT PATIENT_ID) FROM elaprase_patient_payer) AS patients;

In [0]:
CREATE OR REPLACE TEMP VIEW elaprase_patient_insurance AS
SELECT
  e.PATIENT_ID,
  p.PAYER_ID,
  p.INSURANCE_GROUP,
  e.territory_id,
  e.territory_name,
  e.region_id,
  e.region_name
FROM elaprase_patient_latest_khplan e
JOIN payer_base p
  ON e.KH_PLAN = p.PLAN_ID
;

SELECT COUNT(*) AS NROWS, COUNT(DISTINCT PATIENT_ID) AS NPATS FROM elaprase_patient_insurance;

In [0]:
%sql
-- ============================================================
-- ELAPRASE TOTAL LIVES (Elaprase patients only) - payer + territory
-- TOTAL_LIVES = distinct Elaprase patients (medical/procedure OR paid pharmacy)
-- ============================================================

-- Base patient-payer-territory table already exists:
-- elaprase_patient_payer (from MPSII_TREATMENT_TABLE join payer_base)

CREATE OR REPLACE TEMP VIEW payer_total_lives_elaprase AS
SELECT
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  COUNT(DISTINCT PATIENT_ID) AS TOTAL_LIVES
FROM elaprase_patient_payer
GROUP BY
  PAYER_ID, territory_id, territory_name, region_id, region_name;

-- Show tables
SELECT * FROM payer_total_lives_elaprase;

In [0]:
%sql
-- Check 1: Total across groups should equal 410 (no double counting)
SELECT 'SUM_GROUP_LIVES' AS Metric, SUM(TOTAL_LIVES) AS value
FROM payer_total_lives_elaprase

UNION

-- Check 2: Should also equal total patients
SELECT 'EQUAL_TOTAL_PATIENT' AS Metric, COUNT(*) AS value
FROM elaprase_patient_payer

UNION

-- Check 3: Missing payer info
SELECT 'MISSING_PAYER_VAL' AS Metric, COUNT(*) AS missing_payer
FROM elaprase_patient_payer
WHERE PAYER_ID IS NULL;

In [0]:
select count(distinct patient_id) from total_lives
where payer_insurance_group in ('COMMERCIAL', 'MEDICARE', 'MEDICAID');

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_total_lives_by_insurance AS
SELECT
  pb.PAYER_ID,
  tl.territory_id,
  tl.territory_name,
  tl.region_id,
  tl.region_name,

  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='MEDICARE' THEN tl.PATIENT_ID END) AS LIVES_MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='MEDICAID' THEN tl.PATIENT_ID END) AS LIVES_MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='COMMERCIAL' THEN tl.PATIENT_ID END) AS LIVES_COMMERCIAL_PATIENTS,

  COUNT(DISTINCT CASE
    WHEN pb.INSURANCE_GROUP IS NULL
      OR UPPER(TRIM(pb.INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    THEN tl.PATIENT_ID END) AS LIVES_OTHER_PATIENTS

FROM total_lives tl
JOIN payer_base pb
  ON tl.KH_PLAN_ID = pb.PLAN_ID
GROUP BY
  pb.PAYER_ID, tl.territory_id, tl.territory_name, tl.region_id, tl.region_name;

-- Show summary
SELECT SUM(LIVES_MEDICARE_PATIENTS) + SUM(LIVES_MEDICAID_PATIENTS) + SUM(LIVES_COMMERCIAL_PATIENTS) FROM payer_total_lives_by_insurance;


In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN pb.PLAN_ID IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_payer_map,
  SUM(CASE WHEN pb.PLAN_ID IS NULL THEN 1 ELSE 0 END) AS rows_without_payer_map
FROM total_lives tl
LEFT JOIN payer_base pb
  ON tl.KH_PLAN_id = pb.PLAN_ID;   -- use the correct column name from total_lives

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hcp_hco_counts_terr AS
SELECT
  p.PAYER_ID,
  t.territory_id,
  t.territory_name,
  t.region_id,
  t.region_name,
  COUNT(DISTINCT t.HCP_NPI) AS TOTAL_HCPS,
  COUNT(DISTINCT t.HCO_NPI) AS TOTAL_HCOS
FROM MPSII_TREATMENT_TABLE t
JOIN payer_base p
  ON t.KH_PLAN = p.PLAN_ID
GROUP BY
  p.PAYER_ID, t.territory_id, t.territory_name, t.region_id, t.region_name;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW elaprase_patients_by_insurance AS
SELECT
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,

  COUNT(DISTINCT CASE WHEN UPPER(TRIM(INSURANCE_GROUP))='MEDICARE'   THEN PATIENT_ID END) AS MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(INSURANCE_GROUP))='MEDICAID'   THEN PATIENT_ID END) AS MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(INSURANCE_GROUP))='COMMERCIAL' THEN PATIENT_ID END) AS COMMERCIAL_PATIENTS,

  COUNT(DISTINCT CASE
    WHEN INSURANCE_GROUP IS NULL
      OR UPPER(TRIM(INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    THEN PATIENT_ID END) AS OTHER_PATIENTS

FROM elaprase_patient_insurance
GROUP BY
  PAYER_ID, territory_id, territory_name, region_id, region_name;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_total_lives_by_insurance AS
SELECT
  pb.PAYER_ID,
  tl.territory_id,
  tl.territory_name,
  tl.region_id,
  tl.region_name,

  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='MEDICARE' THEN tl.PATIENT_ID END) AS LIVES_MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='MEDICAID' THEN tl.PATIENT_ID END) AS LIVES_MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='COMMERCIAL' THEN tl.PATIENT_ID END) AS LIVES_COMMERCIAL_PATIENTS,

  COUNT(DISTINCT CASE
    WHEN pb.INSURANCE_GROUP IS NULL
      OR UPPER(TRIM(pb.INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    THEN tl.PATIENT_ID END) AS LIVES_OTHER_PATIENTS

FROM total_lives tl
JOIN payer_base pb
  ON tl.KH_PLAN_id = pb.PLAN_ID
GROUP BY
  pb.PAYER_ID, tl.territory_id, tl.territory_name, tl.region_id, tl.region_name;


## wHAT'S THE PURPOSE OF 'ELAPRASE INITIATION'? -- new patients

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW first_elaprase_event AS
SELECT
  PATIENT_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  MIN(FILL_DATE) AS FIRST_FILL_DATE
FROM MPSII_TREATMENT_TABLE
GROUP BY
  PATIENT_ID, territory_id, territory_name, region_id, region_name;


In [0]:
%sql
-- ============================================================
-- 4. Patient Age at Initiation
-- ============================================================
CREATE OR REPLACE TEMP VIEW patient_current_age AS
SELECT
    f.PATIENT_ID,
    YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS AGE_AT_START
FROM first_elaprase_event f
JOIN com_edp_prd.com_raw.kom_patient_demographics d
  ON f.PATIENT_ID = d.PATIENT_ID
WHERE d.PATIENT_YOB IS NOT NULL;



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW total_elaprase_patients AS
SELECT
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  COUNT(DISTINCT PATIENT_ID) AS TOTAL_ELAPRASE_PATIENTS
FROM elaprase_patient_payer
GROUP BY
  PAYER_ID, territory_id, territory_name, region_id, region_name;

-- VALIDATION
SELECT SUM(TOTAL_ELAPRASE_PATIENTS) FROM total_elaprase_patients;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW new_elaprase_patients AS
SELECT
  e.PAYER_ID,
  e.territory_id,
  e.territory_name,
  e.region_id,
  e.region_name,
  COUNT(DISTINCT CASE
    WHEN f.FIRST_FILL_DATE >= DATEADD(month, -1, DATE('${end_date}'))
    THEN f.PATIENT_ID END) AS NEW_ELAPRASE_PATIENTS_R1M,
  COUNT(DISTINCT CASE
    WHEN f.FIRST_FILL_DATE >= DATEADD(month, -3, DATE('${end_date}'))
    THEN f.PATIENT_ID END) AS NEW_ELAPRASE_PATIENTS_R3M
FROM first_elaprase_event f
JOIN elaprase_patient_payer e
  ON f.PATIENT_ID = e.PATIENT_ID
 AND f.territory_id = e.territory_id
GROUP BY
  e.PAYER_ID, e.territory_id, e.territory_name, e.region_id, e.region_name;

-- VALIDATION:
SELECT SUM(NEW_ELAPRASE_PATIENTS_R3M) FROM new_elaprase_patients; 


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW age_buckets AS
SELECT
  e.PAYER_ID,
  e.territory_id,
  e.territory_name,
  e.region_id,
  e.region_name,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START < 5 THEN e.PATIENT_ID END)               AS AGE_LT_5_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START BETWEEN 5 AND 10 THEN e.PATIENT_ID END) AS AGE_5_TO_10_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START BETWEEN 11 AND 18 THEN e.PATIENT_ID END)AS AGE_11_TO_18_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START > 18 THEN e.PATIENT_ID END)             AS AGE_GT_18_YRS
FROM patient_current_age a
JOIN elaprase_patient_payer e
  ON a.PATIENT_ID = e.PATIENT_ID
GROUP BY
  e.PAYER_ID, e.territory_id, e.territory_name, e.region_id, e.region_name;


In [0]:
%python
# %sql
# -- ============================================================
# -- 8. HCP & HCO Counts (ONLY Elaprase Treatments)
# -- ============================================================
# CREATE OR REPLACE TEMP VIEW hcp_hco_counts AS
# SELECT
#     p.PAYER_ID,
#     COUNT(DISTINCT t.HCP_NPI) AS TOTAL_HCPS,
#     COUNT(DISTINCT h.HCO_ID)  AS TOTAL_HCOS
# FROM MPSII_TREATMENT_TABLE t
# JOIN payer_base p
#   ON t.KH_PLAN = p.PLAN_ID
# LEFT JOIN com_edp_prd.com_raw.kom_hcp_hco_xref h
#   ON t.HCP_NPI = h.HCP_NPI
# GROUP BY p.PAYER_ID;

In [0]:
%sql
-- ============================================================
-- 12. Elaprase Pharmacy Claims (Claims Metrics) - PHARMACY ONLY
--     Output view: elaprase_paid_claims
--     Fixes:
--       1) Removed trailing comma issue
--       2) Added UNKNOWN fallback for territory/region
--       3) Used payer_plan_map to avoid row explosion
-- ============================================================


-- ------------------------------------------------------------
-- 0) Create clean PLAN -> PAYER mapping (prevents double counting)
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW payer_plan_map AS
SELECT DISTINCT
  PAYER_ID,
  PLAN_ID
FROM payer_base
WHERE PLAN_ID IS NOT NULL
  AND PAYER_ID IS NOT NULL;



-- ------------------------------------------------------------
-- A) Elaprase PHARMACY claims (distinct pharmacy event ids) + territory/region
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW elaprase_pharmacy_claims_terr AS
WITH prov AS (
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5) AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
)
SELECT DISTINCT
  COALESCE(pe.PRIMARY_KH_PLAN_ID, pe.SECONDARY_KH_PLAN_ID) AS PLAN_ID,
  pe.PHARMACY_EVENT_ID                                     AS EVENT_ID,
  pe.TRANSACTION_RESULT,

  COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
  COALESCE(z.territory_name, 'UNKNOWN') AS territory_name,
  COALESCE(TRY_CAST(z.region_id AS BIGINT), -2) AS region_id,
  COALESCE(z.region_name,    'UNKNOWN') AS region_name

FROM com_edp_prd.com_raw.kom_pharmacy_events pe

LEFT JOIN prov p
  ON TRIM(CAST(pe.PRESCRIBER_NPI AS STRING)) = p.npi_str

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5_int = z.zipcode

WHERE pe.FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
  AND pe.NDC11 IN ('54092070001','540920700')
  AND COALESCE(pe.PRIMARY_KH_PLAN_ID, pe.SECONDARY_KH_PLAN_ID) IS NOT NULL;



-- ------------------------------------------------------------
-- B) Plan-level pharmacy metrics
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW plan_TOTAL_CLAIMS AS
SELECT
  PLAN_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  COUNT(DISTINCT EVENT_ID) AS TOTAL_CLAIMS
FROM elaprase_pharmacy_claims_terr
GROUP BY PLAN_ID, territory_id, territory_name, region_id, region_name;


CREATE OR REPLACE TEMP VIEW plan_approved_fills AS
SELECT
  PLAN_ID,
  territory_id,
  COUNT(DISTINCT EVENT_ID) AS approved_fills
FROM elaprase_pharmacy_claims_terr
WHERE UPPER(TRANSACTION_RESULT) = 'PAID'
GROUP BY PLAN_ID, territory_id;


CREATE OR REPLACE TEMP VIEW plan_rejected_fills AS
SELECT
  PLAN_ID,
  territory_id,
  COUNT(DISTINCT EVENT_ID) AS rejected_fills
FROM elaprase_pharmacy_claims_terr
WHERE UPPER(TRANSACTION_RESULT) = 'REJECTED'
GROUP BY PLAN_ID, territory_id;


CREATE OR REPLACE TEMP VIEW plan_reversed_fills AS
SELECT
  PLAN_ID,
  territory_id,
  COUNT(DISTINCT EVENT_ID) AS reversed_fills
FROM elaprase_pharmacy_claims_terr
WHERE UPPER(TRANSACTION_RESULT) = 'REVERSED'
GROUP BY PLAN_ID, territory_id;



-- ------------------------------------------------------------
-- C) Final payer-level rollup (PHARMACY ONLY) - payer + territory
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW elaprase_paid_claims AS
WITH plan_level AS (
  SELECT
    t.PLAN_ID,
    t.territory_id,
    t.territory_name,
    t.region_id,
    t.region_name,

    COALESCE(t.TOTAL_CLAIMS, 0) AS TOTAL_CLAIMS,
    COALESCE(a.approved_fills, 0)        AS approved_fills,
    COALESCE(r.rejected_fills, 0)        AS rejected_fills,
    COALESCE(v.reversed_fills, 0)        AS reversed_fills

  FROM plan_TOTAL_CLAIMS t

  LEFT JOIN plan_approved_fills a
    ON t.PLAN_ID = a.PLAN_ID
   AND t.territory_id = a.territory_id

  LEFT JOIN plan_rejected_fills r
    ON t.PLAN_ID = r.PLAN_ID
   AND t.territory_id = r.territory_id

  LEFT JOIN plan_reversed_fills v
    ON t.PLAN_ID = v.PLAN_ID
   AND t.territory_id = v.territory_id
)

SELECT
  ppm.PAYER_ID,
  pl.territory_id,
  pl.territory_name,
  pl.region_id,
  pl.region_name,

  SUM(pl.TOTAL_CLAIMS) AS TOTAL_CLAIMS,
  SUM(pl.approved_fills)        AS APPROVED_FILLS,
  SUM(pl.rejected_fills)        AS REJECTED_FILLS,
  SUM(pl.reversed_fills)        AS REVERSED_FILLS,

  ROUND(
    CASE
      WHEN SUM(pl.TOTAL_CLAIMS) = 0 THEN 0
      ELSE 100.0 * SUM(pl.rejected_fills) / SUM(pl.TOTAL_CLAIMS)
    END
  , 2) AS ELAPRASE_REJECTION_RATE,

  ROUND(
    CASE
      WHEN SUM(pl.TOTAL_CLAIMS) = 0 THEN 0
      ELSE 100.0 * SUM(pl.approved_fills) / SUM(pl.TOTAL_CLAIMS)
    END
  , 2) AS ELAPRASE_APPROVAL_RATE

FROM plan_level pl

JOIN payer_plan_map ppm
  ON pl.PLAN_ID = ppm.PLAN_ID

GROUP BY
  ppm.PAYER_ID,
  pl.territory_id,
  pl.territory_name,
  pl.region_id,
  pl.region_name;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_name_mapping AS
SELECT DISTINCT
  PAYER_NAME,
  CASE
    /* United / Optum */
    WHEN UPPER(PAYER_NAME) LIKE '%UNITED%'
      OR UPPER(PAYER_NAME) LIKE '%OPTUM%'
      OR UPPER(PAYER_NAME) LIKE '%UHC%'
      THEN 'United / Optum / Emisar'

    /* Anthem / Elevance */
    WHEN UPPER(PAYER_NAME) LIKE '%ANTHEM%'
      OR UPPER(PAYER_NAME) LIKE '%WELLPOINT%'
      THEN 'Anthem / Elevance / Carelon'

    /* Aetna / CVS */
    WHEN UPPER(PAYER_NAME) LIKE '%AETNA%'
      OR UPPER(PAYER_NAME) LIKE '%CVS%'
      OR UPPER(PAYER_NAME) LIKE '%SILVERSCRIPT%'
      THEN 'Aetna / CVS / Zinc'

    /* Cigna / Express Scripts */
    WHEN UPPER(PAYER_NAME) LIKE '%EXPRESS SCRIPTS%'
      THEN 'Cigna / ESI / Evernorth'

    /* Prime / HCSC */
    WHEN UPPER(PAYER_NAME) LIKE '%PRIME THERAPEUTICS%'
      OR UPPER(PAYER_NAME) LIKE '%HCSC%'
      THEN 'Prime Therapeutics / HCSC'

    /* Molina */
    WHEN UPPER(PAYER_NAME) LIKE '%MOLINA%'
      THEN 'Molina'

    /* Centene */
    WHEN UPPER(PAYER_NAME) LIKE '%WELLCARE%'
      OR UPPER(PAYER_NAME) LIKE '%MERIDIAN%'
      THEN 'Centene'

    /* Humana */
    WHEN UPPER(PAYER_NAME) LIKE '%HUMANA%'
      THEN 'Humana'

    /* Kaiser */
    WHEN UPPER(PAYER_NAME) LIKE '%KAISER%'
      THEN 'Kaiser'

    /* Tricare */
    WHEN UPPER(PAYER_NAME) LIKE '%TRICARE%'
      THEN 'Tricare'

    /* Navitus */
    WHEN UPPER(PAYER_NAME) LIKE '%NAVITUS%'
      THEN 'Navitus'

    /* BCBS – State-based mapping */
    WHEN UPPER(PAYER_NAME) LIKE '%BLUECROSS%'
      OR UPPER(PAYER_NAME) LIKE '%BLUE CROSS%'
      OR UPPER(PAYER_NAME) LIKE '%BCBS%'
      THEN 'Prime Therapeutics / HCSC'

    /* Default */
    ELSE 'UNMAPPED'
  END AS PAYER_ACCOUNT_NAME
FROM payer_base;


In [0]:
%python
# %sql
# -- ============================================================
# -- 13. FINAL PAYER360 MASTER TABLE
# -- ============================================================
# CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS
# SELECT DISTINCT
#     pb.PAYER_ID,
#     pb.PAYER_NAME,
#     DENSE_RANK() OVER (ORDER BY pl.TOTAL_LIVES DESC) AS PAYER_RANK,
#     pb.PARENT_ID,
#     pb.PARENT_NAME,

#     epi.MEDICARE_PATIENTS,
#     epi.MEDICAID_PATIENTS,
#     epi.COMMERCIAL_PATIENTS,

#     tep.TOTAL_ELAPRASE_PATIENTS,
#     np.NEW_ELAPRASE_PATIENTS_R1M,
#     np.NEW_ELAPRASE_PATIENTS_R3M,

#     -- hhc.TOTAL_HCPS,
#     -- hhc.TOTAL_HCOS,

#     pc.TOTAL_CLAIMS,
#     pc.APPROVED_FILLS,
#     pc.REJECTED_FILLS,
#     pc.REVERSED_FILLS,
#     pc.ELAPRASE_REJECTION_RATE,

#     ab.AGE_LT_5_YRS,
#     ab.AGE_5_TO_10_YRS,
#     ab.AGE_11_TO_18_YRS,
#     ab.AGE_GT_18_YRS,

    
  
#     i.PIE_COMPLETED  AS PIE_COMPLETED,
#     i.ACCOUNT_DIRECTOR AS ACCOUNT_DIRECTOR

# FROM payer_base pb
# LEFT JOIN payer_total_lives pl      ON pb.PAYER_ID = pl.PAYER_ID
# LEFT JOIN total_elaprase_patients tep ON pb.PAYER_ID = tep.PAYER_ID
# LEFT JOIN new_elaprase_patients np  ON pb.PAYER_ID = np.PAYER_ID
# -- LEFT JOIN hcp_hco_counts hhc        ON pb.PAYER_ID = hhc.PAYER_ID
# LEFT JOIN elaprase_paid_claims pc   ON pb.PAYER_ID = pc.PAYER_ID
# LEFT JOIN age_buckets ab            ON pb.PAYER_ID = ab.PAYER_ID
# LEFT JOIN elaprase_patients_by_insurance epi  ON pb.PAYER_ID = epi.PAYER_ID
# LEFT JOIN payer_name_mapping m
#   ON pb.PAYER_NAME = m.PAYER_NAME
# LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
#   ON m.PAYER_ACCOUNT_NAME = i.PAYER_ACCOUNT_NAME;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_total_lives AS
SELECT
  p.PAYER_ID,
  tl.territory_id,
  tl.territory_name,
  tl.region_id,
  tl.region_name,
  COUNT(DISTINCT tl.PATIENT_ID) AS TOTAL_LIVES
FROM total_lives tl
JOIN payer_base p
  ON tl.KH_PLAN_id = p.PLAN_ID
GROUP BY
  p.PAYER_ID,
  tl.territory_id,
  tl.territory_name,
  tl.region_id,
  tl.region_name;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_territory_keys AS
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM payer_total_lives
UNION
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM total_elaprase_patients
UNION
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM new_elaprase_patients
UNION
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM elaprase_paid_claims
UNION
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM age_buckets
UNION
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM elaprase_patients_by_insurance
UNION
SELECT DISTINCT PAYER_ID, territory_id, territory_name, region_id, region_name FROM hcp_hco_counts_terr;


In [0]:
%sql
-- ============================================================
-- PAYER DIMENSION (DEDUPED)
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_base_dedup AS
SELECT
  PAYER_ID,
  MAX(PAYER_NAME)  AS PAYER_NAME,
  MAX(PARENT_ID)   AS PARENT_ID,
  MAX(PARENT_NAME) AS PARENT_NAME
FROM payer_base
GROUP BY PAYER_ID;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer360_master_terr AS
SELECT
    k.PAYER_ID,
    pb.PAYER_NAME,

    DENSE_RANK() OVER (
      PARTITION BY k.territory_id
      ORDER BY COALESCE(pl.TOTAL_LIVES, 0) DESC
    ) AS PAYER_RANK,

    pb.PARENT_ID,
    pb.PARENT_NAME,

  -- TOTAL_LIVES = All covered lives (medical + paid pharmacy)

    COALESCE(pl.TOTAL_LIVES, 0) AS TOTAL_LIVES,

    -- Elaprase Insurance Split (Elaprase patients only)
    COALESCE(li.MEDICARE_PATIENTS, 0)   AS MEDICARE_PATIENTS,
    COALESCE(li.MEDICAID_PATIENTS, 0)   AS MEDICAID_PATIENTS,
    COALESCE(li.COMMERCIAL_PATIENTS, 0) AS COMMERCIAL_PATIENTS,
    COALESCE(li.OTHER_PATIENTS, 0)      AS OTHER_PATIENTS,

    COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0) AS TOTAL_ELAPRASE_PATIENTS,

    -- Market share as % (within territory)
    ROUND(
      CASE
        WHEN SUM(COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0)) OVER (PARTITION BY k.territory_id) = 0 THEN 0
        ELSE 100.0 * COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0)
             / SUM(COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0)) OVER (PARTITION BY k.territory_id)
      END
    , 2) AS PAYER_MARKET_SHARE,

    COALESCE(np.NEW_ELAPRASE_PATIENTS_R1M, 0) AS NEW_ELAPRASE_PATIENTS_R1M,
    COALESCE(np.NEW_ELAPRASE_PATIENTS_R3M, 0) AS NEW_ELAPRASE_PATIENTS_R3M,

    COALESCE(hh.TOTAL_HCPS, 0) AS TOTAL_HCPS,
    COALESCE(hh.TOTAL_HCOS, 0) AS TOTAL_HCOS,

    -- Claims Metrics (pharmacy only)
    COALESCE(pc.TOTAL_CLAIMS, 0) AS TOTAL_CLAIMS,
    COALESCE(pc.APPROVED_FILLS, 0)        AS APPROVED_FILLS,
    COALESCE(pc.REJECTED_FILLS, 0)        AS REJECTED_FILLS,
    COALESCE(pc.REVERSED_FILLS, 0)        AS REVERSED_FILLS,
    COALESCE(pc.ELAPRASE_REJECTION_RATE, 0) AS ELAPRASE_REJECTION_RATE,
    COALESCE(pc.ELAPRASE_APPROVAL_RATE, 0)  AS ELAPRASE_APPROVAL_RATE,

    COALESCE(ab.AGE_LT_5_YRS, 0)      AS AGE_LT_5_YRS,
    COALESCE(ab.AGE_5_TO_10_YRS, 0)   AS AGE_5_TO_10_YRS,
    COALESCE(ab.AGE_11_TO_18_YRS, 0)  AS AGE_11_TO_18_YRS,
    COALESCE(ab.AGE_GT_18_YRS, 0)     AS AGE_GT_18_YRS,

    i.PIE_COMPLETED     AS PIE_COMPLETED,
    i.ACCOUNT_DIRECTOR  AS ACCOUNT_DIRECTOR,

    k.territory_id,
    k.territory_name,
    k.region_id,
    k.region_name

FROM payer_territory_keys k

JOIN payer_base_dedup pb
  ON k.PAYER_ID = pb.PAYER_ID

LEFT JOIN payer_total_lives pl
  ON k.PAYER_ID = pl.PAYER_ID
 AND k.territory_id = pl.territory_id


LEFT JOIN elaprase_patients_by_insurance li
  ON k.PAYER_ID = li.PAYER_ID
 AND k.territory_id = li.territory_id

LEFT JOIN total_elaprase_patients tep
  ON k.PAYER_ID = tep.PAYER_ID
 AND k.territory_id = tep.territory_id

LEFT JOIN new_elaprase_patients np
  ON k.PAYER_ID = np.PAYER_ID
 AND k.territory_id = np.territory_id

LEFT JOIN elaprase_paid_claims pc
  ON k.PAYER_ID = pc.PAYER_ID
 AND k.territory_id = pc.territory_id

LEFT JOIN age_buckets ab
  ON k.PAYER_ID = ab.PAYER_ID
 AND k.territory_id = ab.territory_id

LEFT JOIN hcp_hco_counts_terr hh
  ON k.PAYER_ID = hh.PAYER_ID
 AND k.territory_id = hh.territory_id

LEFT JOIN payer_name_mapping m
  ON pb.PAYER_NAME = m.PAYER_NAME

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
  ON m.PAYER_ACCOUNT_NAME = i.PAYER_ACCOUNT_NAME;


# National Level 

In [0]:
%sql
-- ============================================================
-- 0. BASE PAYER TABLE (NATIONAL VERSION)
--    Same as your payer_base, just renamed
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_base_national AS
SELECT DISTINCT
    PAYER_ID,
    PAYER_NAME,
    PARENT_ID,
    PARENT_NAME,
    KH_PLAN_ID      AS PLAN_ID,
    INSURANCE_GROUP
FROM com_edp_prd.com_raw.kom_plans
WHERE PAYER_ID IS NOT NULL;



-- ============================================================
-- N1. TOTAL LIVES BASE (same columns as territory total_lives, but NO geo)
-- ============================================================
-- ============================================================
-- TOTAL LIVES NATIONAL (ALL PATIENTS)
-- ============================================================

CREATE OR REPLACE TEMP VIEW total_lives_national AS

WITH medical AS (
  SELECT DISTINCT
      PATIENT_ID,
      KH_PLAN_ID AS KH_PLAN
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

pharmacy AS (
  SELECT DISTINCT
      PATIENT_ID,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
)

SELECT DISTINCT *
FROM (
  SELECT * FROM medical
  UNION
  SELECT * FROM pharmacy
)
WHERE KH_PLAN IS NOT NULL;




-- ============================================================
-- N1b. PAYER TOTAL LIVES (payer only)
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_total_lives_national AS
SELECT
  p.PAYER_ID,
  COUNT(DISTINCT tl.PATIENT_ID) AS TOTAL_LIVES
FROM total_lives_national tl
JOIN payer_base_national p
  ON tl.KH_PLAN = p.PLAN_ID
GROUP BY p.PAYER_ID;

-- ============================================================
-- N2. MPSII TREATMENT TABLE (same columns as territory version, but NO geo)
-- ============================================================
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE_NATIONAL AS
WITH base AS (
  SELECT *
  FROM (
      -- Medical (Elaprase NDC)
      SELECT DISTINCT
          m.PATIENT_ID                                  AS PATIENT_ID,
          COALESCE(m.RENDERING_NPI, m.REFERRING_NPI)      AS HCP_NPI,
          m.BILLING_NPI                                 AS HCO_NPI,
          m.NDC11                                       AS CODE,
          m.MEDICAL_EVENT_ID                            AS EVENT_ID,
          m.SERVICE_DATE                                AS FILL_DATE,
          m.PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
          m.KH_PLAN_ID                                  AS KH_PLAN,
          NULL                                        AS PHARMACY_CHANNEL,
          'MEDICAL_EVENTS'                            AS TABLE_NAME
      FROM com_edp_prd.com_raw.kom_medical_events m
      INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON m.PATIENT_ID = pat.PATIENT_ID
      WHERE m.NDC11 IN ('54092070001','540920700')

      UNION

      -- Pharmacy (Elaprase NDC | Paid only)
      SELECT DISTINCT
          ph.PATIENT_ID                                  AS PATIENT_ID,
          ph.PRESCRIBER_NPI                              AS HCP_NPI,
          ph.PHARMACY_NPI                                AS HCO_NPI,
          ph.NDC11                                       AS CODE,
          ph.PHARMACY_EVENT_ID                           AS EVENT_ID,
          ph.FILL_DATE                                   AS FILL_DATE,
          NULL                                        AS PLACE_OF_SERVICE,
          COALESCE(ph.PRIMARY_KH_PLAN_ID, ph.SECONDARY_KH_PLAN_ID) AS KH_PLAN,
          ph.PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
          'PHARMACY_EVENTS'                           AS TABLE_NAME
      FROM com_edp_prd.com_raw.kom_pharmacy_events ph
      INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON ph.PATIENT_ID = pat.PATIENT_ID
      WHERE ph.NDC11 IN ('54092070001','540920700')
        AND ph.TRANSACTION_RESULT = 'PAID'

      UNION

      -- Medical (Elaprase Procedure codes)
      SELECT DISTINCT
          m.PATIENT_ID                                  AS PATIENT_ID,
          m.RENDERING_NPI                               AS HCP_NPI,
          m.BILLING_NPI                                 AS HCO_NPI,
          m.PROCEDURE_CODE                              AS CODE,
          m.MEDICAL_EVENT_ID                            AS EVENT_ID,
          m.SERVICE_DATE                                AS FILL_DATE,
          m.PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
          m.KH_PLAN_ID                                  AS KH_PLAN,
          NULL                                        AS PHARMACY_CHANNEL,
          'MEDICAL_EVENTS'                            AS TABLE_NAME
      FROM com_edp_prd.com_raw.kom_medical_events m
      INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON m.PATIENT_ID = pat.PATIENT_ID
      WHERE m.PROCEDURE_CODE IN (
          '99601','99602','96365','96366','J1743','S9357','S9379',
          '38206','38230','38232','38240','38241','38242','38243','38250'
      )
  ) t
  WHERE FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
)
SELECT * FROM base;



-- ============================================================
-- N3. patient->payer (payer only)
-- ============================================================
CREATE OR REPLACE TEMP VIEW elaprase_patient_payer_national AS
SELECT DISTINCT
  t.PATIENT_ID,
  p.PAYER_ID,
  t.FILL_DATE
FROM MPSII_TREATMENT_TABLE_NATIONAL t
JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID;



-- ============================================================
-- N4. total elaprase patients (payer only)
-- ============================================================
CREATE OR REPLACE TEMP VIEW total_elaprase_patients_national AS
SELECT
  PAYER_ID,
  COUNT(DISTINCT PATIENT_ID) AS TOTAL_ELAPRASE_PATIENTS
FROM elaprase_patient_payer_national
GROUP BY PAYER_ID;



-- ============================================================
-- N5. first elaprase event (payer+patient only)
-- ============================================================
CREATE OR REPLACE TEMP VIEW first_elaprase_event_national AS
SELECT
  PAYER_ID,
  PATIENT_ID,
  MIN(FILL_DATE) AS FIRST_FILL_DATE
FROM elaprase_patient_payer_national
GROUP BY PAYER_ID, PATIENT_ID;



-- ============================================================
-- N6. new patients (payer only)
-- ============================================================
CREATE OR REPLACE TEMP VIEW new_elaprase_patients_national AS
SELECT
  PAYER_ID,
  COUNT(DISTINCT CASE
    WHEN FIRST_FILL_DATE >= DATEADD(month, -1, DATE('${end_date}')) THEN PATIENT_ID
  END) AS NEW_ELAPRASE_PATIENTS_R1M,
  COUNT(DISTINCT CASE
    WHEN FIRST_FILL_DATE >= DATEADD(month, -3, DATE('${end_date}')) THEN PATIENT_ID
  END) AS NEW_ELAPRASE_PATIENTS_R3M
FROM first_elaprase_event_national
GROUP BY PAYER_ID;



-- ============================================================
-- N7. patient age at start (payer+patient)
-- ============================================================
CREATE OR REPLACE TEMP VIEW patient_current_age_national AS
SELECT
  f.PAYER_ID,
  f.PATIENT_ID,
  YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS AGE_AT_START
FROM first_elaprase_event_national f
JOIN com_edp_prd.com_raw.kom_patient_demographics d
  ON f.PATIENT_ID = d.PATIENT_ID
WHERE d.PATIENT_YOB IS NOT NULL;



-- ============================================================
-- N8. age buckets (payer only)
-- ============================================================
CREATE OR REPLACE TEMP VIEW age_buckets_national AS
SELECT
  PAYER_ID,
  COUNT(DISTINCT CASE WHEN AGE_AT_START < 5 THEN PATIENT_ID END)                AS AGE_LT_5_YRS,
  COUNT(DISTINCT CASE WHEN AGE_AT_START BETWEEN 5 AND 10 THEN PATIENT_ID END)   AS AGE_5_TO_10_YRS,
  COUNT(DISTINCT CASE WHEN AGE_AT_START BETWEEN 11 AND 18 THEN PATIENT_ID END)  AS AGE_11_TO_18_YRS,
  COUNT(DISTINCT CASE WHEN AGE_AT_START > 18 THEN PATIENT_ID END)               AS AGE_GT_18_YRS
FROM patient_current_age_national
GROUP BY PAYER_ID;



-- ============================================================
-- N9. insurance mix (payer only) - mirrors your structure but no geo
-- ============================================================
CREATE OR REPLACE TEMP VIEW elaprase_patient_insurance_national AS
SELECT DISTINCT
  t.PATIENT_ID,
  p.PAYER_ID,
  p.INSURANCE_GROUP
FROM MPSII_TREATMENT_TABLE_NATIONAL t
JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID;

-- ============================================================
-- N9b. HCP/HCO Counts (NATIONAL)
-- ============================================================
CREATE OR REPLACE TEMP VIEW hcp_hco_counts_national AS
SELECT
  p.PAYER_ID,
  COUNT(DISTINCT t.HCP_NPI) AS TOTAL_HCPS,
  COUNT(DISTINCT t.HCO_NPI) AS TOTAL_HCOS
FROM MPSII_TREATMENT_TABLE_NATIONAL t
JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID
GROUP BY p.PAYER_ID;

CREATE OR REPLACE TEMP VIEW elaprase_patients_by_insurance_national AS
SELECT
  PAYER_ID,

  COUNT(DISTINCT CASE WHEN UPPER(TRIM(INSURANCE_GROUP))='MEDICARE'   THEN PATIENT_ID END) AS MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(INSURANCE_GROUP))='MEDICAID'   THEN PATIENT_ID END) AS MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(INSURANCE_GROUP))='COMMERCIAL' THEN PATIENT_ID END) AS COMMERCIAL_PATIENTS,

  COUNT(DISTINCT CASE
    WHEN INSURANCE_GROUP IS NULL
      OR UPPER(TRIM(INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    THEN PATIENT_ID END) AS OTHER_PATIENTS

FROM elaprase_patient_insurance_national
GROUP BY PAYER_ID;

-- ============================================================
-- N10. pharmacy claims (payer only) - keep percent + formatted
-- ============================================================

-- ============================================================
-- 12N. Elaprase Pharmacy Claims (Claims Metrics) - PHARMACY ONLY (NATIONAL / PAYER-ONLY)
--     Final output view name: elaprase_paid_claims_national
--     SAME COLUMNS as elaprase_paid_claims, but payer-level only
-- ============================================================

-- ------------------------------------------------------------
-- A) Elaprase PHARMACY claims (distinct pharmacy event ids) - NATIONAL
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW elaprase_pharmacy_claims_national AS
SELECT DISTINCT
  COALESCE(pe.PRIMARY_KH_PLAN_ID, pe.SECONDARY_KH_PLAN_ID) AS PLAN_ID,
  pe.PHARMACY_EVENT_ID                                     AS EVENT_ID,
  pe.TRANSACTION_RESULT
FROM com_edp_prd.com_raw.kom_pharmacy_events pe
WHERE pe.FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
  AND pe.NDC11 IN ('54092070001','540920700')
  AND COALESCE(pe.PRIMARY_KH_PLAN_ID, pe.SECONDARY_KH_PLAN_ID) IS NOT NULL;


-- ------------------------------------------------------------
-- B) Plan-level pharmacy metrics - NATIONAL
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW plan_TOTAL_CLAIMS_national AS
SELECT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS TOTAL_CLAIMS
FROM elaprase_pharmacy_claims_national
GROUP BY PLAN_ID;

CREATE OR REPLACE TEMP VIEW plan_approved_fills_national AS
SELECT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS approved_fills
FROM elaprase_pharmacy_claims_national
WHERE UPPER(TRANSACTION_RESULT) = 'PAID'
GROUP BY PLAN_ID;

CREATE OR REPLACE TEMP VIEW plan_rejected_fills_national AS
SELECT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS rejected_fills
FROM elaprase_pharmacy_claims_national
WHERE UPPER(TRANSACTION_RESULT) = 'REJECTED'
GROUP BY PLAN_ID;

CREATE OR REPLACE TEMP VIEW plan_reversed_fills_national AS
SELECT
  PLAN_ID,
  COUNT(DISTINCT EVENT_ID) AS reversed_fills
FROM elaprase_pharmacy_claims_national
WHERE UPPER(TRANSACTION_RESULT) = 'REVERSED'
GROUP BY PLAN_ID;


-- ------------------------------------------------------------
-- C) Final payer-level rollup (PHARMACY ONLY) - NATIONAL
--     SAME OUTPUT COLUMNS as territory elaprase_paid_claims
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW elaprase_paid_claims_national AS
WITH plan_level AS (
  SELECT
    t.PLAN_ID,
    COALESCE(t.TOTAL_CLAIMS, 0) AS TOTAL_CLAIMS,
    COALESCE(a.approved_fills, 0)        AS approved_fills,
    COALESCE(r.rejected_fills, 0)        AS rejected_fills,
    COALESCE(v.reversed_fills, 0)        AS reversed_fills
  FROM plan_TOTAL_CLAIMS_national t
  LEFT JOIN plan_approved_fills_national a
    ON t.PLAN_ID = a.PLAN_ID
  LEFT JOIN plan_rejected_fills_national r
    ON t.PLAN_ID = r.PLAN_ID
  LEFT JOIN plan_reversed_fills_national v
    ON t.PLAN_ID = v.PLAN_ID
)
SELECT
  p.PAYER_ID,

  -- hardcode geo ONLY here (so columns match the territory output)
  -1 AS territory_id,
  'ALL TERRITORIES' AS territory_name,
  -1 AS region_id,
  'ALL TERRITORIES' AS region_name,

  SUM(pl.TOTAL_CLAIMS) AS TOTAL_CLAIMS,
  SUM(pl.approved_fills)        AS APPROVED_FILLS,
  SUM(pl.rejected_fills)        AS REJECTED_FILLS,
  SUM(pl.reversed_fills)        AS REVERSED_FILLS,

  ROUND(
    CASE
      WHEN SUM(pl.TOTAL_CLAIMS) = 0 THEN 0
      ELSE 100.0 * SUM(pl.rejected_fills) / SUM(pl.TOTAL_CLAIMS)
    END
  , 2) AS ELAPRASE_REJECTION_RATE,

  ROUND(
    CASE
      WHEN SUM(pl.TOTAL_CLAIMS) = 0 THEN 0
      ELSE 100.0 * SUM(pl.approved_fills) / SUM(pl.TOTAL_CLAIMS)
    END
  , 2) AS ELAPRASE_APPROVAL_RATE

FROM plan_level pl
JOIN payer_base_national p
  ON pl.PLAN_ID = p.PLAN_ID
GROUP BY p.PAYER_ID;




-- ============================================================
-- N11. national keys (payer + NATIONAL geo hardcode)
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer360_master_national_keys AS
SELECT DISTINCT
  PAYER_ID,

  -- overwrite for dashboard display
  'ALL TERRITORIES' AS PAYER_NAME,
  PARENT_ID,
  'ALL TERRITORIES' AS PARENT_NAME,

  CAST(-1 AS BIGINT) AS territory_id,
  'ALL TERRITORIES'  AS territory_name,
  CAST(-1 AS BIGINT) AS region_id,
  'ALL TERRITORIES'  AS region_name

FROM payer_base_national;

-- ============================================================
-- N12. final national master (payer only + NATIONAL geo)
-- ============================================================

CREATE OR REPLACE TEMP VIEW payer360_master_national AS
SELECT
  k.PAYER_ID,
  pb.PAYER_NAME,

  DENSE_RANK() OVER (ORDER BY COALESCE(pl.TOTAL_LIVES, 0) DESC) AS PAYER_RANK,

  pb.PARENT_ID,
  pb.PARENT_NAME,

  COALESCE(pl.TOTAL_LIVES, 0) AS TOTAL_LIVES,

  COALESCE(epi.MEDICARE_PATIENTS, 0)   AS MEDICARE_PATIENTS,
  COALESCE(epi.MEDICAID_PATIENTS, 0)   AS MEDICAID_PATIENTS,
  COALESCE(epi.COMMERCIAL_PATIENTS, 0) AS COMMERCIAL_PATIENTS,
  COALESCE(epi.OTHER_PATIENTS, 0)      AS OTHER_PATIENTS,

  COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0) AS TOTAL_ELAPRASE_PATIENTS,

  ROUND(
    CASE
      WHEN SUM(COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0)) OVER () = 0 THEN 0
      ELSE 100.0 * COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0)
           / SUM(COALESCE(tep.TOTAL_ELAPRASE_PATIENTS, 0)) OVER ()
    END
  , 2) AS PAYER_MARKET_SHARE,

  COALESCE(np.NEW_ELAPRASE_PATIENTS_R1M, 0) AS NEW_ELAPRASE_PATIENTS_R1M,
  COALESCE(np.NEW_ELAPRASE_PATIENTS_R3M, 0) AS NEW_ELAPRASE_PATIENTS_R3M,

  COALESCE(hh.TOTAL_HCPS, 0) AS TOTAL_HCPS,
  COALESCE(hh.TOTAL_HCOS, 0) AS TOTAL_HCOS,

  COALESCE(pc.TOTAL_CLAIMS, 0) AS TOTAL_CLAIMS,
  COALESCE(pc.APPROVED_FILLS, 0)        AS APPROVED_FILLS,
  COALESCE(pc.REJECTED_FILLS, 0)        AS REJECTED_FILLS,
  COALESCE(pc.REVERSED_FILLS, 0)        AS REVERSED_FILLS,
  COALESCE(pc.ELAPRASE_REJECTION_RATE, 0) AS ELAPRASE_REJECTION_RATE,
  COALESCE(pc.ELAPRASE_APPROVAL_RATE, 0)  AS ELAPRASE_APPROVAL_RATE,

  COALESCE(ab.AGE_LT_5_YRS, 0)      AS AGE_LT_5_YRS,
  COALESCE(ab.AGE_5_TO_10_YRS, 0)   AS AGE_5_TO_10_YRS,
  COALESCE(ab.AGE_11_TO_18_YRS, 0)  AS AGE_11_TO_18_YRS,
  COALESCE(ab.AGE_GT_18_YRS, 0)     AS AGE_GT_18_YRS,

  i.PIE_COMPLETED     AS PIE_COMPLETED,
  i.ACCOUNT_DIRECTOR  AS ACCOUNT_DIRECTOR,

  k.territory_id,
  k.territory_name,
  k.region_id,
  k.region_name

FROM payer360_master_national_keys k

JOIN payer_base_dedup pb
  ON k.PAYER_ID = pb.PAYER_ID

LEFT JOIN payer_total_lives_national pl
  ON k.PAYER_ID = pl.PAYER_ID

LEFT JOIN total_elaprase_patients_national tep
  ON k.PAYER_ID = tep.PAYER_ID

LEFT JOIN new_elaprase_patients_national np
  ON k.PAYER_ID = np.PAYER_ID

LEFT JOIN elaprase_paid_claims_national pc
  ON k.PAYER_ID = pc.PAYER_ID

LEFT JOIN age_buckets_national ab
  ON k.PAYER_ID = ab.PAYER_ID

LEFT JOIN elaprase_patients_by_insurance_national epi
  ON k.PAYER_ID = epi.PAYER_ID

LEFT JOIN hcp_hco_counts_national hh
  ON k.PAYER_ID = hh.PAYER_ID

LEFT JOIN payer_name_mapping m
  ON pb.PAYER_NAME = m.PAYER_NAME

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
  ON m.PAYER_ACCOUNT_NAME = i.PAYER_ACCOUNT_NAME;



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payer_total_lives_by_insurance_national AS
SELECT
  pb.PAYER_ID,

  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='MEDICARE' THEN tl.PATIENT_ID END) AS LIVES_MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='MEDICAID' THEN tl.PATIENT_ID END) AS LIVES_MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(pb.INSURANCE_GROUP))='COMMERCIAL' THEN tl.PATIENT_ID END) AS LIVES_COMMERCIAL_PATIENTS,

  COUNT(DISTINCT CASE
    WHEN pb.INSURANCE_GROUP IS NULL
      OR UPPER(TRIM(pb.INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    THEN tl.PATIENT_ID END) AS LIVES_OTHER_PATIENTS

FROM total_lives_national tl
JOIN payer_base_national pb
  ON tl.KH_PLAN = pb.PLAN_ID
GROUP BY pb.PAYER_ID;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hcp_hco_counts_national AS
SELECT
  p.PAYER_ID,
  COUNT(DISTINCT t.HCP_NPI) AS TOTAL_HCPS,
  COUNT(DISTINCT t.HCO_NPI) AS TOTAL_HCOS
FROM MPSII_TREATMENT_TABLE_NATIONAL t
JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID
GROUP BY p.PAYER_ID;


In [0]:
%sql
-- ============================================================
-- ALL PAYERS PHARMACY CLAIMS ROLLUP (Territory + National)
-- ============================================================

-- -------------------------------
-- A) Territory-level pharmacy rollup (ALL PAYERS)
-- -------------------------------
CREATE OR REPLACE TEMP VIEW elaprase_pharmacy_claims_rollup_terr AS
WITH prov AS (
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5) AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
)
SELECT DISTINCT
  pe.PHARMACY_EVENT_ID AS EVENT_ID,
  pe.TRANSACTION_RESULT,

  COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
  COALESCE(z.territory_name, 'UNKNOWN') AS territory_name,
  COALESCE(TRY_CAST(z.region_id AS BIGINT), -2) AS region_id,
  COALESCE(z.region_name, 'UNKNOWN') AS region_name

FROM com_edp_prd.com_raw.kom_pharmacy_events pe

LEFT JOIN prov p
  ON TRIM(CAST(pe.PRESCRIBER_NPI AS STRING)) = p.npi_str

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5_int = z.zipcode

WHERE pe.FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
  AND pe.NDC11 IN ('54092070001','540920700');



CREATE OR REPLACE TEMP VIEW elaprase_paid_claims_rollup_terr AS
SELECT
  'ALL TERRITORIES' AS PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,

  COUNT(DISTINCT EVENT_ID) AS TOTAL_CLAIMS,

  COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'PAID' THEN EVENT_ID END)     AS APPROVED_FILLS,
  COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'REJECTED' THEN EVENT_ID END) AS REJECTED_FILLS,
  COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'REVERSED' THEN EVENT_ID END) AS REVERSED_FILLS,

  ROUND(
    CASE
      WHEN COUNT(DISTINCT EVENT_ID) = 0 THEN 0
      ELSE 100.0 * COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'REJECTED' THEN EVENT_ID END)
           / COUNT(DISTINCT EVENT_ID)
    END
  , 2) AS ELAPRASE_REJECTION_RATE,

  ROUND(
    CASE
      WHEN COUNT(DISTINCT EVENT_ID) = 0 THEN 0
      ELSE 100.0 * COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'PAID' THEN EVENT_ID END)
           / COUNT(DISTINCT EVENT_ID)
    END
  , 2) AS ELAPRASE_APPROVAL_RATE

FROM elaprase_pharmacy_claims_rollup_terr
GROUP BY
  territory_id, territory_name, region_id, region_name;



-- -------------------------------
-- B) National pharmacy rollup (ALL PAYERS)
-- -------------------------------
CREATE OR REPLACE TEMP VIEW elaprase_pharmacy_claims_rollup_national AS
SELECT DISTINCT
  pe.PHARMACY_EVENT_ID AS EVENT_ID,
  pe.TRANSACTION_RESULT
FROM com_edp_prd.com_raw.kom_pharmacy_events pe
WHERE pe.FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
  AND pe.NDC11 IN ('54092070001','540920700');



CREATE OR REPLACE TEMP VIEW elaprase_paid_claims_rollup_national AS
SELECT
  'ALL TERRITORIES' AS PAYER_ID,

  CAST(-1 AS BIGINT) AS territory_id,
  'ALL TERRITORIES'  AS territory_name,
  CAST(-1 AS BIGINT) AS region_id,
  'ALL TERRITORIES'  AS region_name,

  COUNT(DISTINCT EVENT_ID) AS TOTAL_CLAIMS,

  COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'PAID' THEN EVENT_ID END)     AS APPROVED_FILLS,
  COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'REJECTED' THEN EVENT_ID END) AS REJECTED_FILLS,
  COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'REVERSED' THEN EVENT_ID END) AS REVERSED_FILLS,

  ROUND(
    CASE
      WHEN COUNT(DISTINCT EVENT_ID) = 0 THEN 0
      ELSE 100.0 * COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'REJECTED' THEN EVENT_ID END)
           / COUNT(DISTINCT EVENT_ID)
    END
  , 2) AS ELAPRASE_REJECTION_RATE,

  ROUND(
    CASE
      WHEN COUNT(DISTINCT EVENT_ID) = 0 THEN 0
      ELSE 100.0 * COUNT(DISTINCT CASE WHEN UPPER(TRANSACTION_RESULT) = 'PAID' THEN EVENT_ID END)
           / COUNT(DISTINCT EVENT_ID)
    END
  , 2) AS ELAPRASE_APPROVAL_RATE

FROM elaprase_pharmacy_claims_rollup_national;


In [0]:
CREATE OR REPLACE TEMP VIEW payer360_rollup_all_payers AS

-- ============================================================
-- A) TERRITORY LEVEL ROLLUP (ALL PAYERS COMBINED)
-- ============================================================

SELECT
  -1 AS PAYER_ID,
  'ALL TERRITORIES' AS PAYER_NAME,
  -1 AS PAYER_RANK,
  -1 AS PARENT_ID,
  'ALL TERRITORIES' AS PARENT_NAME,

  -- Insurance split (Elaprase only)
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(p.INSURANCE_GROUP))='MEDICARE' THEN t.PATIENT_ID END) AS MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(p.INSURANCE_GROUP))='MEDICAID' THEN t.PATIENT_ID END) AS MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(p.INSURANCE_GROUP))='COMMERCIAL' THEN t.PATIENT_ID END) AS COMMERCIAL_PATIENTS,
  COUNT(DISTINCT CASE
      WHEN p.INSURANCE_GROUP IS NULL
        OR UPPER(TRIM(p.INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
      THEN t.PATIENT_ID END) AS OTHER_PATIENTS,

  -- ✅ TOTAL_LIVES (ALL PATIENTS, NOT Elaprase)
  SUM(pl.TOTAL_LIVES) AS TOTAL_LIVES,

  -- Elaprase patients
  COUNT(DISTINCT t.PATIENT_ID) AS TOTAL_ELAPRASE_PATIENTS,

  -1 AS PAYER_MARKET_SHARE,

  COUNT(DISTINCT CASE
      WHEN f.FIRST_FILL_DATE >= DATEADD(month, -1, DATE('${end_date}'))
      THEN f.PATIENT_ID END) AS NEW_ELAPRASE_PATIENTS_R1M,

  COUNT(DISTINCT CASE
      WHEN f.FIRST_FILL_DATE >= DATEADD(month, -3, DATE('${end_date}'))
      THEN f.PATIENT_ID END) AS NEW_ELAPRASE_PATIENTS_R3M,

  COUNT(DISTINCT t.HCP_NPI) AS TOTAL_HCPS,
  COUNT(DISTINCT t.HCO_NPI) AS TOTAL_HCOS,

  COALESCE(pc.TOTAL_CLAIMS,0) AS TOTAL_CLAIMS,
  COALESCE(pc.APPROVED_FILLS,0) AS APPROVED_FILLS,
  COALESCE(pc.REJECTED_FILLS,0) AS REJECTED_FILLS,
  COALESCE(pc.REVERSED_FILLS,0) AS REVERSED_FILLS,
  COALESCE(pc.ELAPRASE_REJECTION_RATE,0) AS ELAPRASE_REJECTION_RATE,
  COALESCE(pc.ELAPRASE_APPROVAL_RATE,0) AS ELAPRASE_APPROVAL_RATE,

  COUNT(DISTINCT CASE WHEN a.AGE_AT_START < 5 THEN t.PATIENT_ID END) AS AGE_LT_5_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START BETWEEN 5 AND 10 THEN t.PATIENT_ID END) AS AGE_5_TO_10_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START BETWEEN 11 AND 18 THEN t.PATIENT_ID END) AS AGE_11_TO_18_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START > 18 THEN t.PATIENT_ID END) AS AGE_GT_18_YRS,

  'ALL TERRITORIES' AS PIE_COMPLETED,
  'ALL TERRITORIES' AS ACCOUNT_DIRECTOR,

  t.territory_id,
  t.territory_name,
  t.region_id,
  t.region_name

FROM MPSII_TREATMENT_TABLE t
LEFT JOIN payer_base p
  ON t.KH_PLAN = p.PLAN_ID
LEFT JOIN payer_total_lives pl
  ON t.territory_id = pl.territory_id
LEFT JOIN first_elaprase_event f
  ON t.PATIENT_ID = f.PATIENT_ID
LEFT JOIN patient_current_age a
  ON t.PATIENT_ID = a.PATIENT_ID
LEFT JOIN elaprase_paid_claims_rollup_terr pc
  ON t.territory_id = pc.territory_id

GROUP BY
  t.territory_id,
  t.territory_name,
  t.region_id,
  t.region_name,
  pc.TOTAL_CLAIMS,
  pc.APPROVED_FILLS,
  pc.REJECTED_FILLS,
  pc.REVERSED_FILLS,
  pc.ELAPRASE_REJECTION_RATE,
  pc.ELAPRASE_APPROVAL_RATE


UNION ALL


-- ============================================================
-- B) NATIONAL LEVEL ROLLUP
-- ============================================================

SELECT
  -1 AS PAYER_ID,
  'ALL TERRITORIES' AS PAYER_NAME,
  -1 AS PAYER_RANK,
  -1 AS PARENT_ID,
  'ALL TERRITORIES' AS PARENT_NAME,

  COUNT(DISTINCT CASE WHEN UPPER(TRIM(p.INSURANCE_GROUP))='MEDICARE' THEN t.PATIENT_ID END) AS MEDICARE_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(p.INSURANCE_GROUP))='MEDICAID' THEN t.PATIENT_ID END) AS MEDICAID_PATIENTS,
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(p.INSURANCE_GROUP))='COMMERCIAL' THEN t.PATIENT_ID END) AS COMMERCIAL_PATIENTS,
  COUNT(DISTINCT CASE
      WHEN p.INSURANCE_GROUP IS NULL
        OR UPPER(TRIM(p.INSURANCE_GROUP)) NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
      THEN t.PATIENT_ID END) AS OTHER_PATIENTS,

  -- ✅ TOTAL_LIVES (ALL PATIENTS)
  SUM(pl.TOTAL_LIVES) AS TOTAL_LIVES,

  COUNT(DISTINCT t.PATIENT_ID) AS TOTAL_ELAPRASE_PATIENTS,

  -1 AS PAYER_MARKET_SHARE,

  COUNT(DISTINCT CASE
      WHEN f.FIRST_FILL_DATE >= DATEADD(month, -1, DATE('${end_date}'))
      THEN f.PATIENT_ID END) AS NEW_ELAPRASE_PATIENTS_R1M,

  COUNT(DISTINCT CASE
      WHEN f.FIRST_FILL_DATE >= DATEADD(month, -3, DATE('${end_date}'))
      THEN f.PATIENT_ID END) AS NEW_ELAPRASE_PATIENTS_R3M,

  COUNT(DISTINCT t.HCP_NPI) AS TOTAL_HCPS,
  COUNT(DISTINCT t.HCO_NPI) AS TOTAL_HCOS,

  COALESCE(pc.TOTAL_CLAIMS,0) AS TOTAL_CLAIMS,
  COALESCE(pc.APPROVED_FILLS,0) AS APPROVED_FILLS,
  COALESCE(pc.REJECTED_FILLS,0) AS REJECTED_FILLS,
  COALESCE(pc.REVERSED_FILLS,0) AS REVERSED_FILLS,
  COALESCE(pc.ELAPRASE_REJECTION_RATE,0) AS ELAPRASE_REJECTION_RATE,
  COALESCE(pc.ELAPRASE_APPROVAL_RATE,0) AS ELAPRASE_APPROVAL_RATE,

  COUNT(DISTINCT CASE WHEN a.AGE_AT_START < 5 THEN t.PATIENT_ID END) AS AGE_LT_5_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START BETWEEN 5 AND 10 THEN t.PATIENT_ID END) AS AGE_5_TO_10_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START BETWEEN 11 AND 18 THEN t.PATIENT_ID END) AS AGE_11_TO_18_YRS,
  COUNT(DISTINCT CASE WHEN a.AGE_AT_START > 18 THEN t.PATIENT_ID END) AS AGE_GT_18_YRS,

  'ALL TERRITORIES' AS PIE_COMPLETED,
  'ALL TERRITORIES' AS ACCOUNT_DIRECTOR,

  CAST(-1 AS BIGINT) AS territory_id,
  'ALL TERRITORIES'  AS territory_name,
  CAST(-1 AS BIGINT) AS region_id,
  'ALL TERRITORIES'  AS region_name

FROM MPSII_TREATMENT_TABLE_NATIONAL t
LEFT JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID
LEFT JOIN payer_total_lives_national pl
  ON 1=1
LEFT JOIN first_elaprase_event_national f
  ON t.PATIENT_ID = f.PATIENT_ID
LEFT JOIN patient_current_age_national a
  ON t.PATIENT_ID = a.PATIENT_ID
LEFT JOIN elaprase_paid_claims_rollup_national pc
  ON 1=1

GROUP BY
  pc.TOTAL_CLAIMS,
  pc.APPROVED_FILLS,
  pc.REJECTED_FILLS,
  pc.REVERSED_FILLS,
  pc.ELAPRASE_REJECTION_RATE,
  pc.ELAPRASE_APPROVAL_RATE;


In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS

SELECT *
FROM (

    SELECT
      PAYER_ID,
      PAYER_NAME,
      PAYER_RANK,
      PARENT_ID,
      PARENT_NAME,
      MEDICARE_PATIENTS,
      MEDICAID_PATIENTS,
      COMMERCIAL_PATIENTS,
      OTHER_PATIENTS,
      TOTAL_LIVES,
      TOTAL_ELAPRASE_PATIENTS,
      PAYER_MARKET_SHARE,
      NEW_ELAPRASE_PATIENTS_R1M,
      NEW_ELAPRASE_PATIENTS_R3M,
      TOTAL_HCPS,
      TOTAL_HCOS,
      TOTAL_CLAIMS,
      APPROVED_FILLS,
      REJECTED_FILLS,
      REVERSED_FILLS,
      ELAPRASE_REJECTION_RATE,
      ELAPRASE_APPROVAL_RATE,
      AGE_LT_5_YRS,
      AGE_5_TO_10_YRS,
      AGE_11_TO_18_YRS,
      AGE_GT_18_YRS,
      COALESCE(PIE_COMPLETED, '-')     AS PIE_COMPLETED,
      COALESCE(ACCOUNT_DIRECTOR, '-')  AS ACCOUNT_DIRECTOR,
      territory_id,
      territory_name,
      region_id,
      region_name
    FROM payer360_master_terr

    UNION ALL
    SELECT * FROM payer360_rollup_all_payers

    UNION ALL
    SELECT
      PAYER_ID,
      PAYER_NAME,
      PAYER_RANK,
      PARENT_ID,
      PARENT_NAME,
      MEDICARE_PATIENTS,
      MEDICAID_PATIENTS,
      COMMERCIAL_PATIENTS,
      OTHER_PATIENTS,
      TOTAL_LIVES,
      TOTAL_ELAPRASE_PATIENTS,
      PAYER_MARKET_SHARE,
      NEW_ELAPRASE_PATIENTS_R1M,
      NEW_ELAPRASE_PATIENTS_R3M,
      TOTAL_HCPS,
      TOTAL_HCOS,
      TOTAL_CLAIMS,
      APPROVED_FILLS,
      REJECTED_FILLS,
      REVERSED_FILLS,
      ELAPRASE_REJECTION_RATE,
      ELAPRASE_APPROVAL_RATE,
      AGE_LT_5_YRS,
      AGE_5_TO_10_YRS,
      AGE_11_TO_18_YRS,
      AGE_GT_18_YRS,
      COALESCE(PIE_COMPLETED, '-')     AS PIE_COMPLETED,
      COALESCE(ACCOUNT_DIRECTOR, '-')  AS ACCOUNT_DIRECTOR,
      territory_id,
      territory_name,
      region_id,
      region_name
    FROM payer360_master_national

) final_data

WHERE
    COALESCE(TOTAL_ELAPRASE_PATIENTS,0) > 0
    OR
    COALESCE(TOTAL_CLAIMS,0) > 0;


In [0]:
%sql
SELECT 
*
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master;

In [0]:
%sql
SELECT
  COUNT(*) total_rows,
  COUNT(DISTINCT concat(PAYER_ID,'|',territory_name)) distinct_keys
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master_sample;


In [0]:
%sql
SELECT COUNT(DISTINCT PATIENT_ID)
FROM MPSII_TREATMENT_TABLE;


In [0]:
%sql
SELECT
  SUM(TOTAL_LIVES),
  SUM(TOTAL_ELAPRASE_PATIENTS)
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master;


In [0]:
%sql
SELECT PAYER_NAME, COUNT(DISTINCT TOTAL_ELAPRASE_PATIENTS) 
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
GROUP BY  1
ORDER BY 2 DESC;

In [0]:
%sql
SELECT
  territory_name,
  COUNT(*) AS rows
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
GROUP BY territory_name
ORDER BY rows DESC;

# HCO Details Table

In [0]:
%sql
-- ============================================================
-- PAYER 360 - HCO DETAIL TABLE
-- Grain: 1 row per PAYER_ID + TERRITORY_NAME + HCO_ID
-- Includes BOTH Territory-level + National-level records
-- ============================================================

-- ------------------------------------------------------------
-- 1) HCO DIMENSION (HCO_ID -> HCO_NAME)
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW hco_dim AS
SELECT
  TRIM(CAST(NPI AS STRING)) AS HCO_ID,
  MAX(ORGANIZATION_NAME)    AS HCO_NAME
FROM com_edp_prd.com_raw.kom_providers
WHERE NPI IS NOT NULL
  AND PROVIDER_TYPE = 'ORGANIZATION'
GROUP BY TRIM(CAST(NPI AS STRING));


-- ------------------------------------------------------------
-- 2) HCO DETAIL - TERRITORY LEVEL
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW payer_360_hco_detail_terr AS
SELECT
  p.PAYER_ID,

  t.territory_id,
  t.territory_name,
  t.region_id,
  t.region_name,

  TRIM(CAST(t.HCO_NPI AS STRING)) AS HCO_ID,
  hd.HCO_NAME,

  COUNT(DISTINCT t.PATIENT_ID) AS PATIENT_COUNT,
  COUNT(DISTINCT t.EVENT_ID)   AS CLAIMS_COUNT,
  MAX(t.FILL_DATE)             AS LAST_TREATMENT_DATE

FROM MPSII_TREATMENT_TABLE t
JOIN payer_base p
  ON t.KH_PLAN = p.PLAN_ID

LEFT JOIN hco_dim hd
  ON TRIM(CAST(t.HCO_NPI AS STRING)) = hd.HCO_ID

WHERE t.HCO_NPI IS NOT NULL

GROUP BY
  p.PAYER_ID,
  t.territory_id,
  t.territory_name,
  t.region_id,
  t.region_name,
  TRIM(CAST(t.HCO_NPI AS STRING)),
  hd.HCO_NAME;


-- ------------------------------------------------------------
-- 3) HCO DETAIL - NATIONAL LEVEL
-- ------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW payer_360_hco_detail_national AS
SELECT
  p.PAYER_ID,

  CAST(-1 AS BIGINT) AS territory_id,
  'ALL TERRITORIES'  AS territory_name,
  CAST(-1 AS BIGINT) AS region_id,
  'ALL TERRITORIES'  AS region_name,

  TRIM(CAST(t.HCO_NPI AS STRING)) AS HCO_ID,
  hd.HCO_NAME,

  COUNT(DISTINCT t.PATIENT_ID) AS PATIENT_COUNT,
  COUNT(DISTINCT t.EVENT_ID)   AS CLAIMS_COUNT,
  MAX(t.FILL_DATE)             AS LAST_TREATMENT_DATE

FROM MPSII_TREATMENT_TABLE_NATIONAL t
JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID

LEFT JOIN hco_dim hd
  ON TRIM(CAST(t.HCO_NPI AS STRING)) = hd.HCO_ID

WHERE t.HCO_NPI IS NOT NULL

GROUP BY
  p.PAYER_ID,
  TRIM(CAST(t.HCO_NPI AS STRING)),
  hd.HCO_NAME;


-- ------------------------------------------------------------
-- 4) FINAL UNION TABLE CREATION
-- ------------------------------------------------------------
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_360_hco_detail AS

SELECT
  CAST(PAYER_ID AS STRING) AS PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  HCO_ID,
  HCO_NAME,
  coalesce (CAST(PATIENT_COUNT AS BIGINT), 0) AS PATIENT_COUNT,
  coalesce (CAST(CLAIMS_COUNT  AS BIGINT),0) AS CLAIMS_COUNT,
  CAST(LAST_TREATMENT_DATE AS DATE) AS LAST_TREATMENT_DATE
FROM payer_360_hco_detail_terr

UNION ALL

SELECT
  CAST(PAYER_ID AS STRING) AS PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  HCO_ID,
  HCO_NAME,
  coalesce (CAST(PATIENT_COUNT AS BIGINT), 0) AS PATIENT_COUNT,
  coalesce (CAST(CLAIMS_COUNT  AS BIGINT),0) AS CLAIMS_COUNT,
  CAST(LAST_TREATMENT_DATE AS DATE) AS LAST_TREATMENT_DATE
FROM payer_360_hco_detail_national;

-- ============================================================
-- 5. Final HCO Detail Table
-- ============================================================

SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.payer_360_hco_detail;


# HCP Details Table

In [0]:
%sql
-- ============================================================
-- PAYER 360 - HCP DETAIL TABLE
-- Table Name: com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail
-- Grain: 1 row per PAYER_ID + TERRITORY_ID + HCP_ID
-- Includes BOTH Territory-level + National-level records
-- ============================================================


-- ============================================================
-- 1. HCP DIMENSION (HCP_ID -> HCP_NAME + Specialty + ZIP)
-- ============================================================
CREATE OR REPLACE TEMP VIEW hcp_dim AS
SELECT DISTINCT
  TRIM(CAST(NPI AS STRING)) AS HCP_ID,
  TRIM(CONCAT(COALESCE(FIRST_NAME,''),' ',COALESCE(LAST_NAME,''))) AS HCP_NAME,
  PRIMARY_SPECIALTY AS SPECIALTY,
  TRY_CAST(
    SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP, ''), '[^0-9]', ''), 1, 5) AS BIGINT
  ) AS ZIP5
FROM com_edp_prd.com_raw.kom_providers
WHERE NPI IS NOT NULL
  AND PROVIDER_TYPE = 'INDIVIDUAL';


-- ============================================================
-- 2. HCP TERRITORY MAP (ZIP -> Territory/Region)
-- ============================================================
CREATE OR REPLACE TEMP VIEW hcp_geo_map AS
SELECT DISTINCT
  h.HCP_ID,
  COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
  COALESCE(z.territory_name, 'UNKNOWN')            AS territory_name,
  COALESCE(TRY_CAST(z.region_id AS BIGINT), -2)    AS region_id,
  COALESCE(z.region_name, 'UNKNOWN')               AS region_name
FROM hcp_dim h
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON h.ZIP5 = z.zipcode;


-- ============================================================
-- 3. BASE CLAIMS - TERRITORY LEVEL
-- ============================================================
CREATE OR REPLACE TEMP VIEW hcp_elaprase_claims_terr AS
SELECT DISTINCT
  p.PAYER_ID,
  g.territory_id,
  g.territory_name,
  g.region_id,
  g.region_name,

  TRIM(CAST(t.HCP_NPI AS STRING)) AS HCP_ID,
  TRIM(CAST(t.HCO_NPI AS STRING)) AS PRIMARY_HCO_NPI,

  t.PATIENT_ID,
  t.EVENT_ID,
  t.FILL_DATE
FROM MPSII_TREATMENT_TABLE t
JOIN payer_base p
  ON t.KH_PLAN = p.PLAN_ID
LEFT JOIN hcp_geo_map g
  ON TRIM(CAST(t.HCP_NPI AS STRING)) = g.HCP_ID
WHERE t.HCP_NPI IS NOT NULL;


-- ============================================================
-- 4. BASE CLAIMS - NATIONAL LEVEL
-- ============================================================
CREATE OR REPLACE TEMP VIEW hcp_elaprase_claims_national AS
SELECT DISTINCT
  p.PAYER_ID,

  CAST(-1 AS BIGINT) AS territory_id,
  'ALL TERRITORIES'  AS territory_name,
  CAST(-1 AS BIGINT) AS region_id,
  'ALL TERRITORIES'  AS region_name,

  TRIM(CAST(t.HCP_NPI AS STRING)) AS HCP_ID,
  TRIM(CAST(t.HCO_NPI AS STRING)) AS PRIMARY_HCO_NPI,

  t.PATIENT_ID,
  t.EVENT_ID,
  t.FILL_DATE
FROM MPSII_TREATMENT_TABLE_NATIONAL t
JOIN payer_base_national p
  ON t.KH_PLAN = p.PLAN_ID
WHERE t.HCP_NPI IS NOT NULL;


-- ============================================================
-- 5. AGGREGATION - TERRITORY LEVEL
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_360_hcp_detail_terr AS
SELECT
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,

  HCP_ID,

  COUNT(DISTINCT PATIENT_ID) AS PATIENT_COUNT,
  COUNT(DISTINCT EVENT_ID)   AS CLAIMS_COUNT,
  MAX(FILL_DATE)             AS LAST_TREATMENT_DATE
FROM hcp_elaprase_claims_terr
GROUP BY
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  HCP_ID;

-- ============================================================
-- 6. AGGREGATION - NATIONAL LEVEL
-- ============================================================
CREATE OR REPLACE TEMP VIEW payer_360_hcp_detail_national AS
SELECT
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,

  HCP_ID,

  COUNT(DISTINCT PATIENT_ID) AS PATIENT_COUNT,
  COUNT(DISTINCT EVENT_ID)   AS CLAIMS_COUNT,
  MAX(FILL_DATE)             AS LAST_TREATMENT_DATE
FROM hcp_elaprase_claims_national
GROUP BY
  PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,
  HCP_ID;

-- ============================================================
-- 7. FINAL UNION TABLE CREATION
-- ============================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail AS

SELECT DISTINCT
  CAST(PAYER_ID AS STRING) AS PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,

  h.HCP_ID,
  COALESCE(h.HCP_NAME, 'UNKNOWN')  AS HCP_NAME,
  COALESCE(h.SPECIALTY, 'UNKNOWN') AS SPECIALTY,

  COALESCE(ref.HCO_NPI, '-')   AS PRIMARY_HCO_NPI,
  COALESCE(ref.HCO_NAME, 'UNKNOWN') AS HCO_NAME,

  COALESCE(CAST(PATIENT_COUNT AS BIGINT), 0) AS PATIENT_COUNT,
  COALESCE(CAST(CLAIMS_COUNT  AS BIGINT), 0) AS CLAIMS_COUNT,

  CAST(LAST_TREATMENT_DATE AS DATE) AS LAST_TREATMENT_DATE

FROM payer_360_hcp_detail_terr b
LEFT JOIN hcp_dim h
  ON b.HCP_ID = h.HCP_ID
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0219 ref
  ON ref.HCP_NPI = b.HCP_ID

UNION 

SELECT DISTINCT
  CAST(PAYER_ID AS STRING) AS PAYER_ID,
  territory_id,
  territory_name,
  region_id,
  region_name,

  h.HCP_ID,
  COALESCE(h.HCP_NAME, 'UNKNOWN')  AS HCP_NAME,
  COALESCE(h.SPECIALTY, 'UNKNOWN') AS SPECIALTY,

  COALESCE(ref.HCO_NPI, '-') AS PRIMARY_HCO_NPI,
  COALESCE(ref.HCO_NAME, 'UNKNOWN') AS HCO_NAME,

  COALESCE(CAST(PATIENT_COUNT AS BIGINT), 0) AS PATIENT_COUNT,
  COALESCE(CAST(CLAIMS_COUNT  AS BIGINT), 0) AS CLAIMS_COUNT,

  CAST(LAST_TREATMENT_DATE AS DATE) AS LAST_TREATMENT_DATE

FROM payer_360_hcp_detail_national b
LEFT JOIN hcp_dim h
  ON b.HCP_ID = h.HCP_ID
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0219 ref
  ON ref.HCP_NPI = b.HCP_ID;


-- ============================================================
-- 8. Validation Output
-- ============================================================
-- SELECT *
-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail;

select hcp_id, count(hco_name) from payer_360_hcp_detail group by 1 order by 2 desc;


In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail;

In [0]:
select hcp_id, count(hco_name) from com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail group by 1 order by 2 desc;

In [0]:
%sql
create or replace table com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail as 
select a.*, b.payer_name
from com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail as a
left join (select distinct PAYER_ID, PAYER_NAME from com_edp_prd.com_raw.kom_plans
where PAYER_ID is not null and PAYER_NAME is not null) as b on a.PAYER_ID = b.payer_id

In [0]:
%sql
create or replace table com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail as 
select * except(region_id, territory_id), cast(territory_id as string) as territory_id, cast(region_id as string) as region_id
from com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail

In [0]:
%sql
select * from com_edp_prd.cmpa_insights_internal_schema.payer360_master

## Appendix

In [0]:
%sql
-- ============================================================
-- TOTAL LIVES BASE (ALL PATIENTS - NOT patient360 restricted)
-- Medical + PAID Pharmacy
-- ============================================================

CREATE OR REPLACE TEMP VIEW total_lives AS

WITH medical AS (
  SELECT DISTINCT
      PATIENT_ID,
      KH_PLAN_ID AS KH_PLAN,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

pharmacy AS (
  SELECT DISTINCT
      PATIENT_ID,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      PRESCRIBER_NPI AS HCP_NPI
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
),

base AS (
  SELECT * FROM medical
  UNION
  SELECT * FROM pharmacy
),

prov AS (
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    MAX(
      TRY_CAST(
        SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP,''),'[^0-9]',''),1,5)
        AS BIGINT
      )
    ) AS zip5_int
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
  GROUP BY TRIM(CAST(NPI AS STRING))
)

SELECT DISTINCT
  b.PATIENT_ID AS PATIENT_ID,
  b.KH_PLAN,
  COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
  COALESCE(z.territory_name, 'UNKNOWN') AS territory_name,
  COALESCE(TRY_CAST(z.region_id AS BIGINT), -2) AS region_id,
  COALESCE(z.region_name, 'UNKNOWN') AS region_name
FROM base b
LEFT JOIN prov p
  ON TRIM(CAST(b.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5_int = z.zipcode
WHERE b.KH_PLAN IS NOT NULL;


----
SELECT COUNT(*) AS n, COUNT(DISTINCT PATIENT_ID) AS n_distinct FROM total_lives;